# 06 Evaluate Improved WLASL100 Model

This notebook evaluates the improved WLASL100 model trained in `05_train_improved_wlasl100_bigru_attention.ipynb`.

The goal is to understand the model beyond overall accuracy by checking:

- Test Top-1, Top-3, and Top-5 accuracy
- Macro F1-score
- Per-class accuracy
- Per-class Top-5 success
- Best-performing signs
- Worst-performing signs
- Most common class confusions
- High-confidence wrong predictions
- Examples where Top-1 is wrong but Top-5 is correct

This helps decide what to improve before moving to WLASL300.

In [ ]:
from pathlib import Path
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path("E:/Be_My_Ear")

BASE_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / "WLASL100"
CLEAN_INDEX_FILE = BASE_DIR / "wlasl100_clean_keypoint_index.csv"
LABEL_MAP_FILE = PROJECT_ROOT / "data" / "label_maps" / "asl_wlasl100_labels.json"

MODEL_DIR = PROJECT_ROOT / "models" / "ASL"
MODEL_PATH = MODEL_DIR / "improved_bigru_attention_wlasl100.pt"
NORM_STATS_PATH = MODEL_DIR / "wlasl100_train_norm_stats.npz"

EVAL_DIR = PROJECT_ROOT / "reports" / "phase1_wlasl100"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

print("Clean index exists:", CLEAN_INDEX_FILE.exists())
print("Label map exists:", LABEL_MAP_FILE.exists())
print("Model checkpoint exists:", MODEL_PATH.exists())
print("Normalisation stats exists:", NORM_STATS_PATH.exists())
print("Evaluation reports will save to:", EVAL_DIR)

## 1. Load dataset and label map

In [ ]:
df = pd.read_csv(CLEAN_INDEX_FILE)

with open(LABEL_MAP_FILE, "r", encoding="utf-8") as f:
    label_map = json.load(f)

id_to_gloss = {
    int(label_id): info["gloss"]
    for label_id, info in label_map.items()
}

NUM_CLASSES = df["label_id"].nunique()

print("Clean samples:", len(df))
print("Classes:", NUM_CLASSES)
print("First label examples:")
list(id_to_gloss.items())[:10]

## 2. Recreate the same train / validation / test split

The evaluation notebook recreates the same split logic used in the improved training notebook so the test set is consistent.

In [ ]:
train_records = []
val_records = []
test_records = []

for label_id, group in df.groupby("label_id"):
    group = group.sample(frac=1, random_state=SEED).reset_index(drop=True)

    n = len(group)

    n_test = max(1, int(round(n * 0.15)))
    n_val = max(1, int(round(n * 0.15)))

    test_part = group.iloc[:n_test]
    val_part = group.iloc[n_test:n_test + n_val]
    train_part = group.iloc[n_test + n_val:]

    train_records.append(train_part)
    val_records.append(val_part)
    test_records.append(test_part)

train_df = pd.concat(train_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = pd.concat(val_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = pd.concat(test_records).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))

print("Train classes:", train_df["label_id"].nunique())
print("Validation classes:", val_df["label_id"].nunique())
print("Test classes:", test_df["label_id"].nunique())

## 3. Load normalisation statistics

The improved model uses global training-set normalisation. If the saved normalisation file is missing, this notebook recomputes it from the training split.

In [ ]:
def compute_train_normalisation_stats(train_dataframe):
    total_sum = None
    total_sq_sum = None
    total_count = 0

    for path in tqdm(train_dataframe["keypoint_path"], desc="Computing train mean/std"):
        arr = np.load(path).astype(np.float32)

        if total_sum is None:
            total_sum = arr.sum(axis=0)
            total_sq_sum = (arr ** 2).sum(axis=0)
        else:
            total_sum += arr.sum(axis=0)
            total_sq_sum += (arr ** 2).sum(axis=0)

        total_count += arr.shape[0]

    mean = total_sum / total_count
    variance = (total_sq_sum / total_count) - (mean ** 2)
    variance = np.maximum(variance, 1e-6)
    std = np.sqrt(variance)

    return mean.astype(np.float32), std.astype(np.float32)


if NORM_STATS_PATH.exists():
    norm_stats = np.load(NORM_STATS_PATH)
    train_mean = norm_stats["mean"].astype(np.float32)
    train_std = norm_stats["std"].astype(np.float32)
    print("Loaded normalisation stats from:", NORM_STATS_PATH)
else:
    train_mean, train_std = compute_train_normalisation_stats(train_df)
    np.savez(NORM_STATS_PATH, mean=train_mean, std=train_std)
    print("Recomputed and saved normalisation stats to:", NORM_STATS_PATH)

print("Mean shape:", train_mean.shape)
print("Std shape:", train_std.shape)

## 4. Dataset class

This must match the improved model training notebook. It uses normalised keypoints plus velocity features.

In [ ]:
class ImprovedSignKeypointDataset(Dataset):
    def __init__(self, dataframe, mean, std, use_velocity=True):
        self.dataframe = dataframe.reset_index(drop=True)
        self.mean = mean.reshape(1, -1).astype(np.float32)
        self.std = std.reshape(1, -1).astype(np.float32)
        self.use_velocity = use_velocity

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        keypoints = np.load(row["keypoint_path"]).astype(np.float32)

        keypoints = (keypoints - self.mean) / (self.std + 1e-6)

        if self.use_velocity:
            velocity = np.zeros_like(keypoints, dtype=np.float32)
            velocity[1:] = keypoints[1:] - keypoints[:-1]
            features = np.concatenate([keypoints, velocity], axis=1)
        else:
            features = keypoints

        label = int(row["label_id"])

        features = torch.tensor(features, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.long)

        return features, label

In [ ]:
BATCH_SIZE = 32
USE_VELOCITY = True
INPUT_SIZE = 516 if USE_VELOCITY else 258

train_dataset = ImprovedSignKeypointDataset(train_df, train_mean, train_std, use_velocity=USE_VELOCITY)
val_dataset = ImprovedSignKeypointDataset(val_df, train_mean, train_std, use_velocity=USE_VELOCITY)
test_dataset = ImprovedSignKeypointDataset(test_df, train_mean, train_std, use_velocity=USE_VELOCITY)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

x_batch, y_batch = next(iter(test_loader))

print("Test input batch shape:", x_batch.shape)
print("Test label batch shape:", y_batch.shape)

## 5. Recreate model architecture and load checkpoint

The class definition must match the model used in `05_train_improved_wlasl100_bigru_attention.ipynb`.

In [ ]:
class BiGRUAttentionModel(nn.Module):
    def __init__(
        self,
        input_size,
        hidden_size,
        num_classes,
        num_layers=2,
        dropout=0.4
    ):
        super(BiGRUAttentionModel, self).__init__()

        self.input_projection = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.gru = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        x = self.input_projection(x)

        gru_out, _ = self.gru(x)

        attention_scores = self.attention(gru_out).squeeze(-1)
        attention_weights = torch.softmax(attention_scores, dim=1).unsqueeze(-1)

        context = torch.sum(gru_out * attention_weights, dim=1)

        logits = self.classifier(context)

        return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

checkpoint = torch.load(MODEL_PATH, map_location=device)

model = BiGRUAttentionModel(
    input_size=checkpoint.get("input_size", INPUT_SIZE),
    hidden_size=256,
    num_classes=checkpoint.get("num_classes", NUM_CLASSES),
    num_layers=2,
    dropout=0.4
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded checkpoint from:", MODEL_PATH)
print("Best checkpoint epoch:", checkpoint.get("epoch"))
print("Best validation F1:", checkpoint.get("best_val_f1"))
print("Best validation Top-5:", checkpoint.get("best_val_top5"))

## 6. Collect predictions

In [ ]:
def collect_predictions(model, loader, device):
    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Collecting predictions"):
            x = x.to(device)
            y = y.to(device)

            outputs = model(x)
            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_labels.extend(y.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


y_true, y_pred, y_probs = collect_predictions(model, test_loader, device)

print("Predictions collected:", len(y_pred))

## 7. Overall metrics

In [ ]:
def numpy_top_k_accuracy(y_true, y_probs, k):
    correct = 0

    for true_label, prob in zip(y_true, y_probs):
        top_k_preds = np.argsort(prob)[-k:]
        if true_label in top_k_preds:
            correct += 1

    return correct / len(y_true)


test_top1 = accuracy_score(y_true, y_pred)
test_top3 = numpy_top_k_accuracy(y_true, y_probs, k=3)
test_top5 = numpy_top_k_accuracy(y_true, y_probs, k=5)
test_macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

print("=" * 80)
print("Improved WLASL100 Model Evaluation")
print("=" * 80)
print(f"Test Top-1 Accuracy: {test_top1:.4f}")
print(f"Test Top-3 Accuracy: {test_top3:.4f}")
print(f"Test Top-5 Accuracy: {test_top5:.4f}")
print(f"Test Macro F1: {test_macro_f1:.4f}")
print("=" * 80)

In [ ]:
overall_metrics = pd.DataFrame([{
    "dataset": "WLASL100",
    "model": "BiGRU + Temporal Attention",
    "clean_samples": len(df),
    "classes": NUM_CLASSES,
    "test_samples": len(test_df),
    "test_top1_accuracy": test_top1,
    "test_top3_accuracy": test_top3,
    "test_top5_accuracy": test_top5,
    "test_macro_f1": test_macro_f1,
    "best_val_f1": checkpoint.get("best_val_f1"),
    "best_val_top5": checkpoint.get("best_val_top5"),
    "checkpoint_epoch": checkpoint.get("epoch")
}])

OVERALL_METRICS_FILE = EVAL_DIR / "improved_wlasl100_overall_metrics.csv"
overall_metrics.to_csv(OVERALL_METRICS_FILE, index=False)

print("Saved overall metrics to:")
print(OVERALL_METRICS_FILE)

overall_metrics

## 8. Per-class performance

This shows which signs the model recognises well and which signs need improvement.

In [ ]:
per_class_records = []

for label_id in sorted(np.unique(y_true)):
    mask = y_true == label_id

    class_true = y_true[mask]
    class_pred = y_pred[mask]
    class_probs = y_probs[mask]

    top1_acc = accuracy_score(class_true, class_pred)
    top3_acc = numpy_top_k_accuracy(class_true, class_probs, k=3)
    top5_acc = numpy_top_k_accuracy(class_true, class_probs, k=5)

    per_class_records.append({
        "label_id": int(label_id),
        "gloss": id_to_gloss.get(int(label_id), str(label_id)),
        "test_samples": int(mask.sum()),
        "top1_accuracy": top1_acc,
        "top3_accuracy": top3_acc,
        "top5_accuracy": top5_acc
    })

per_class_df = pd.DataFrame(per_class_records)

PER_CLASS_FILE = EVAL_DIR / "improved_wlasl100_per_class_performance.csv"
per_class_df.to_csv(PER_CLASS_FILE, index=False)

print("Saved per-class performance to:")
print(PER_CLASS_FILE)

per_class_df.head()

In [ ]:
print("Best-performing classes by Top-1 accuracy:")
per_class_df.sort_values(
    ["top1_accuracy", "top5_accuracy", "test_samples"],
    ascending=[False, False, False]
).head(15)

In [ ]:
print("Worst-performing classes by Top-1 accuracy:")
per_class_df.sort_values(
    ["top1_accuracy", "top5_accuracy", "test_samples"],
    ascending=[True, True, False]
).head(15)

In [ ]:
print("Classes where Top-1 is weak but Top-5 is strong:")
per_class_df["top5_minus_top1"] = per_class_df["top5_accuracy"] - per_class_df["top1_accuracy"]

per_class_df.sort_values(
    "top5_minus_top1",
    ascending=False
).head(15)

## 9. Visualise per-class accuracy

In [ ]:
sorted_per_class = per_class_df.sort_values("top1_accuracy", ascending=False)

plt.figure(figsize=(16, 5))
plt.bar(sorted_per_class["gloss"], sorted_per_class["top1_accuracy"])
plt.title("WLASL100 Per-Class Top-1 Accuracy")
plt.xlabel("Gloss")
plt.ylabel("Top-1 Accuracy")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
sorted_per_class = per_class_df.sort_values("top5_accuracy", ascending=False)

plt.figure(figsize=(16, 5))
plt.bar(sorted_per_class["gloss"], sorted_per_class["top5_accuracy"])
plt.title("WLASL100 Per-Class Top-5 Accuracy")
plt.xlabel("Gloss")
plt.ylabel("Top-5 Accuracy")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## 10. Confusion matrix and common confusions

A full 100-class confusion matrix is hard to read, so this section also extracts the most common wrong pairs.

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))

plt.figure(figsize=(14, 12))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix - Improved WLASL100")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
confusion_records = []

for true_label in range(NUM_CLASSES):
    for pred_label in range(NUM_CLASSES):
        count = cm[true_label, pred_label]

        if true_label != pred_label and count > 0:
            confusion_records.append({
                "true_label_id": true_label,
                "true_gloss": id_to_gloss.get(true_label, str(true_label)),
                "predicted_label_id": pred_label,
                "predicted_gloss": id_to_gloss.get(pred_label, str(pred_label)),
                "count": int(count)
            })

confusion_df = pd.DataFrame(confusion_records)

if len(confusion_df) > 0:
    confusion_df = confusion_df.sort_values("count", ascending=False)

CONFUSION_FILE = EVAL_DIR / "improved_wlasl100_common_confusions.csv"
confusion_df.to_csv(CONFUSION_FILE, index=False)

print("Saved common confusions to:")
print(CONFUSION_FILE)

confusion_df.head(20)

## 11. Prediction-level analysis

This creates a detailed table for every test prediction, including confidence and Top-5 predictions.

In [ ]:
prediction_records = []
test_df_reset = test_df.reset_index(drop=True)

for i in range(len(y_true)):
    true_id = int(y_true[i])
    pred_id = int(y_pred[i])
    confidence = float(y_probs[i][pred_id])

    top5_ids = np.argsort(y_probs[i])[-5:][::-1]
    top5_probs = y_probs[i][top5_ids]

    top5_glosses = [
        id_to_gloss.get(int(label_id), str(label_id))
        for label_id in top5_ids
    ]

    top5_contains_true = true_id in top5_ids

    prediction_records.append({
        "video_id": test_df_reset.iloc[i]["video_id"],
        "true_label_id": true_id,
        "true_gloss": id_to_gloss.get(true_id, str(true_id)),
        "predicted_label_id": pred_id,
        "predicted_gloss": id_to_gloss.get(pred_id, str(pred_id)),
        "confidence": confidence,
        "correct_top1": true_id == pred_id,
        "correct_top5": top5_contains_true,
        "top5_label_ids": ", ".join([str(int(x)) for x in top5_ids]),
        "top5_glosses": ", ".join(top5_glosses),
        "top5_probabilities": ", ".join([f"{float(p):.4f}" for p in top5_probs]),
        "keypoint_path": test_df_reset.iloc[i]["keypoint_path"],
        "video_path": test_df_reset.iloc[i]["video_path"]
    })

predictions_df = pd.DataFrame(prediction_records)

PREDICTIONS_FILE = EVAL_DIR / "improved_wlasl100_test_predictions.csv"
predictions_df.to_csv(PREDICTIONS_FILE, index=False)

print("Saved prediction-level results to:")
print(PREDICTIONS_FILE)

predictions_df.head()

In [ ]:
print("Correct Top-1 prediction examples:")
predictions_df[predictions_df["correct_top1"] == True].sort_values(
    "confidence",
    ascending=False
).head(10)

In [ ]:
print("Top-1 wrong but Top-5 correct examples:")
predictions_df[
    (predictions_df["correct_top1"] == False) &
    (predictions_df["correct_top5"] == True)
].sort_values("confidence", ascending=False).head(10)

In [ ]:
print("High-confidence wrong predictions:")
predictions_df[
    predictions_df["correct_top1"] == False
].sort_values("confidence", ascending=False).head(15)

In [ ]:
print("Low-confidence predictions:")
predictions_df.sort_values("confidence", ascending=True).head(15)

## 12. Confidence analysis

This helps decide what confidence threshold might be useful later when we build the actual app.

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(predictions_df["confidence"], bins=20)
plt.title("Prediction Confidence Distribution")
plt.xlabel("Confidence")
plt.ylabel("Number of Test Samples")
plt.tight_layout()
plt.show()

In [ ]:
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

threshold_records = []

for threshold in thresholds:
    confident_df = predictions_df[predictions_df["confidence"] >= threshold]

    if len(confident_df) == 0:
        coverage = 0
        top1_at_threshold = np.nan
        top5_at_threshold = np.nan
    else:
        coverage = len(confident_df) / len(predictions_df)
        top1_at_threshold = confident_df["correct_top1"].mean()
        top5_at_threshold = confident_df["correct_top5"].mean()

    threshold_records.append({
        "confidence_threshold": threshold,
        "coverage": coverage,
        "top1_accuracy_on_confident_samples": top1_at_threshold,
        "top5_accuracy_on_confident_samples": top5_at_threshold,
        "num_confident_samples": len(confident_df)
    })

threshold_df = pd.DataFrame(threshold_records)

THRESHOLD_FILE = EVAL_DIR / "improved_wlasl100_confidence_threshold_analysis.csv"
threshold_df.to_csv(THRESHOLD_FILE, index=False)

print("Saved confidence threshold analysis to:")
print(THRESHOLD_FILE)

threshold_df

## 13. Final evaluation summary

In [ ]:
print("Final evaluation summary")
print("------------------------")
print(f"Dataset: WLASL100")
print(f"Clean samples: {len(df)}")
print(f"Test samples: {len(test_df)}")
print(f"Classes: {NUM_CLASSES}")
print(f"Model: BiGRU + Temporal Attention")
print(f"Input shape: (60, {INPUT_SIZE})")
print(f"Test Top-1 Accuracy: {test_top1:.4f}")
print(f"Test Top-3 Accuracy: {test_top3:.4f}")
print(f"Test Top-5 Accuracy: {test_top5:.4f}")
print(f"Test Macro F1: {test_macro_f1:.4f}")
print()
print("Saved report files:")
print("-", OVERALL_METRICS_FILE)
print("-", PER_CLASS_FILE)
print("-", CONFUSION_FILE)
print("-", PREDICTIONS_FILE)
print("-", THRESHOLD_FILE)